# Programmations parallèle & concurrente

## Mesurer avant d'optimiser

La fonction `doublons` ci-dessous renvoie les éléments présents plusieurs fois dans une liste.

1. Avec [`timeit.timeit`](https://docs.python.org/fr/3/library/timeit.html#timeit.timeit), mesurez le temps d'exécution de `doublons(nombres)`.
2. Avec [`cProfile.run`](https://docs.python.org/fr/3/library/profile.html#profile.run), trouvez l'appel dans lequel la fonction passe le plus de temps.
3. Proposez une version plus rapide, vérifiez qu'elle renvoie les mêmes éléments et mesurez le gain avec `timeit`.

In [ ]:
import random

nombres = [random.randrange(5_000) for _ in range(5_000)]


def doublons(valeurs: list[int]) -> list[int]:
  resultat = []
  for valeur in valeurs:
    if valeurs.count(valeur) > 1 and valeur not in resultat:
      resultat.append(valeur)
  return resultat


# Votre code ici

### Solution

In [ ]:
import collections
import cProfile
import timeit

duree = timeit.timeit("doublons(nombres)", globals=globals(), number=3) / 3
print(f"doublons : {duree:.3f}s")

cProfile.run("doublons(nombres)", sort="tottime")


def doublons_rapide(valeurs: list[int]) -> list[int]:
  return [valeur for valeur, n in collections.Counter(valeurs).items() if n > 1]


assert sorted(doublons_rapide(nombres)) == sorted(doublons(nombres))
duree = timeit.timeit("doublons_rapide(nombres)", globals=globals(), number=3) / 3
print(f"doublons_rapide : {duree:.5f}s")

Le profil montre que presque tout le temps est passé dans `list.count`, appelé pour chaque élément : chaque appel reparcourt toute la liste, l'algorithme est donc quadratique. [`collections.Counter`](https://docs.python.org/fr/3/library/collections.html#collections.Counter) compte toutes les occurrences en un seul parcours.

## Appeler parallélement une fonction avec un argument

Appelez la fonction `f` qui donne des infos sur le process qui l'exécute parallélement avec une `multiprocessing.Pool`.

In [ ]:
import os


def f(n: int) -> None:
  print(f"Process {n}")
  print(f"ID du process parent : {os.getppid()}")
  print(f"ID du process : {os.getpid()}")


# Votre code ici

### Solution

In [ ]:
import multiprocessing
import os


def f(n: int) -> None:
  print(f"Process {n}")
  print(f"ID du process parent : {os.getppid()}")
  print(f"ID du process : {os.getpid()}")


with multiprocessing.Pool() as pool:
  pool.map(f, range(10))

## Appeler en parallèle une fonction dont un ou plusieurs arguments sont constants

Utilisez [`functools.partial`](https://docs.python.org/fr/3/library/functools.html#functools.partial) et adaptez le code précédent pour appeler `f` avec des `n` allant de 0 à 9 et `verbose` toujours fixé à `False`.

In [ ]:
import os


def f(n: int, verbose: bool) -> int:
  if verbose:
    print(f"Process {n}")
    print(f"ID du process parent : {os.getppid()}")
    print(f"ID du process : {os.getpid()}")
  return n * 2


# Votre code ici

### Solution

In [ ]:
import functools
import multiprocessing
import os


def f(n: int, verbose: bool) -> int:
  if verbose:
    print(f"Process {n}")
    print(f"ID du process parent : {os.getppid()}")
    print(f"ID du process : {os.getpid()}")
  return n * 2


with multiprocessing.Pool() as pool:
  ns = pool.map(functools.partial(f, verbose=False), range(10))

print(ns)

## Télécharger plusieurs fichiers simultanément, avec un seul argument

Utilisez un `multiprocessing.pool.ThreadPool` pour télécharger plusieurs fichiers simultanément. Dans un premier temps, nous allons télécharger 10 fois la page aléatoire de Wikipédia : `https://en.wikipedia.org/wiki/Special:Random`. La fonction parallélisée récupèrera simplement un entier et stockera le résultat du téléchargement dans un dossier `articles`, sous `{n}.html` si `n` est l'entier.

Pour le téléchargement de fichier, vous pourrez utiliser le code suivant :

```python
request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(request) as response, path.open("wb") as fh:
    shutil.copyfileobj(response, fh)
```

L'en-tête `User-Agent` est nécessaire : Wikipédia, comme d'autres sites, refuse l'agent par défaut de `urllib` (erreur HTTP 403).

où `url` est l'URL à télécharger et `path` est le chemin où écrire le fichier.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
import urllib.request

articles = pathlib.Path("articles")
articles.mkdir(exist_ok=True)
url = "https://en.wikipedia.org/wiki/Special:Random"


def download_article(n: int) -> None:
  # Wikipédia et Python Tutor refusent l'agent par défaut de urllib (erreur 403)
  request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
  with (
    urllib.request.urlopen(request) as response,
    (articles / f"{n}.html").open("wb") as fh,
  ):
    shutil.copyfileobj(response, fh)


with multiprocessing.pool.ThreadPool() as pool:
  results = pool.map(download_article, range(10))

## Télécharger plusieurs fichiers simultanément, avec deux arguments

De manière similaire à l'exercice précédent, on souhaite télécharger plusieurs pages en même temps. Cette fois on souhaite donner à notre worker une url et un chemin où écrire, plutôt que seulement un entier.

Adaptez le code précédent pour télécharger `to_download`.

In [ ]:
import pathlib

downloads = pathlib.Path("downloads")
downloads.mkdir(exist_ok=True)

to_download = (
  ("https://docs.python.org/fr/3/", downloads / "python-docs.html"),
  ("http://pythontutor.com/", downloads / "python-tutor.html"),
  ("https://www.google.com/", downloads / "google.html"),
)

# Votre code ici

### Solution

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
import urllib.request

downloads = pathlib.Path("downloads")
downloads.mkdir(exist_ok=True)

to_download = (
  ("https://docs.python.org/", downloads / "python-docs.html"),
  ("http://pythontutor.com/", downloads / "python-tutor.html"),
  ("https://www.google.com/", downloads / "google.html"),
)


def download(url: str, path: pathlib.Path) -> None:
  # Wikipédia et Python Tutor refusent l'agent par défaut de urllib (erreur 403)
  request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
  with urllib.request.urlopen(request) as response, path.open("wb") as fh:
    shutil.copyfileobj(response, fh)


with multiprocessing.pool.ThreadPool() as pool:
  pool.starmap(download, to_download)

## Téléchargements concurrents avec `asyncio`

Reprenez `to_download` et la fonction `download` de l'exercice précédent, et téléchargez les pages de manière concurrente avec `asyncio` : [`asyncio.to_thread`](https://docs.python.org/fr/3/library/asyncio-task.html#asyncio.to_thread) exécute une fonction bloquante (comme `download`) sans bloquer la boucle d'événements, et [`asyncio.gather`](https://docs.python.org/fr/3/library/asyncio-task.html#asyncio.gather) attend plusieurs coroutines à la fois. Affichez la durée totale des téléchargements.

Attention : dans Colab (comme dans Jupyter), une boucle d'événements tourne déjà, et `asyncio.run(main())` lève donc une `RuntimeError`. Écrivez directement `await main()` dans la cellule ; dans un script, on utiliserait `asyncio.run(main())`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import asyncio
import time


async def main() -> None:
  debut = time.perf_counter()
  await asyncio.gather(
    *(asyncio.to_thread(download, url, path) for url, path in to_download)
  )
  print(f"{len(to_download)} pages téléchargées en {time.perf_counter() - debut:.2f}s")


await main()

## Créer une architecture producteur / consommateur

Le code suivant fait communiquer deux processus par une [`multiprocessing.Queue`](https://docs.python.org/fr/3/library/multiprocessing.html#multiprocessing.Queue) : le producteur y dépose des éléments, le consommateur les retire et les affiche.

Exécutez-le, puis répondez : pourquoi le producteur dépose-t-il `None` à la fin ? Modifiez ensuite le code pour lancer deux consommateurs au lieu d'un.

In [ ]:
import multiprocessing


def consumer(queue: multiprocessing.Queue) -> None:
  while True:
    item = queue.get()
    if item is None:
      break
    print(item)


def producer(queue: multiprocessing.Queue) -> None:
  for i in range(10):
    queue.put(i)
  queue.put(None)


if __name__ == "__main__":
  queue = multiprocessing.Queue()
  consumer = multiprocessing.Process(target=consumer, args=(queue,))
  producer = multiprocessing.Process(target=producer, args=(queue,))
  consumer.start()
  producer.start()
  consumer.join()
  producer.join()

In [ ]:
# Votre code ici

### Solution

`None` est une valeur sentinelle : elle signale au consommateur qu'il n'y aura plus d'éléments. Sans elle, le consommateur resterait bloqué indéfiniment sur `queue.get()`. Avec plusieurs consommateurs, chacun doit recevoir sa propre sentinelle :

In [ ]:
import multiprocessing


def consumer(name: str, queue: multiprocessing.Queue) -> None:
  while True:
    item = queue.get()
    if item is None:
      break
    print(f"{name} : {item}")


def producer(queue: multiprocessing.Queue, n_consumers: int) -> None:
  for i in range(10):
    queue.put(i)
  for _ in range(n_consumers):
    queue.put(None)


if __name__ == "__main__":
  queue = multiprocessing.Queue()
  consumers = [
    multiprocessing.Process(target=consumer, args=(f"consommateur {k}", queue))
    for k in range(2)
  ]
  producer_process = multiprocessing.Process(
    target=producer, args=(queue, len(consumers))
  )
  for process in [*consumers, producer_process]:
    process.start()
  for process in [*consumers, producer_process]:
    process.join()

## Effectuer deux tâches différentes en parallèle avec un `Pool`

Le code suivant soumet deux calculs différents au même `Pool` avec [`map_async`](https://docs.python.org/fr/3/library/multiprocessing.html#multiprocessing.pool.Pool.map_async), qui rend la main immédiatement au lieu d'attendre les résultats comme `map` : les deux calculs s'exécutent donc en même temps sur les processus du pool.

Réécrivez-le avec [`concurrent.futures.ProcessPoolExecutor`](https://docs.python.org/fr/3/library/concurrent.futures.html#concurrent.futures.ProcessPoolExecutor).

In [ ]:
import multiprocessing


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i**2


if __name__ == "__main__":
  with multiprocessing.Pool() as pool:
    a_results = pool.map_async(a, range(10))
    b_results = pool.map_async(b, range(10))
    print(a_results.get())
    print(b_results.get())

In [ ]:
# Votre code ici

### Solution

In [ ]:
import concurrent.futures


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i**2


if __name__ == "__main__":
  with concurrent.futures.ProcessPoolExecutor() as executor:
    a_futures = [executor.submit(a, i) for i in range(10)]
    b_futures = [executor.submit(b, i) for i in range(10)]
    print([future.result() for future in a_futures])
    print([future.result() for future in b_futures])